# Chapter 5: The Long Tail
### *Reserving and IBNR*

Arclight is four years old and writing 1,200 policies. The CFO has called a meeting. The balance sheet shows a healthy premium surplus — but the auditors are asking a pointed question: *how much do you actually owe on claims that have already happened but are not fully settled yet?*

This is the reserving problem. In property insurance, a fire claim is reported within days and settled within weeks. In cyber, the clock runs differently:

- A breach that occurred in Q3 may not be discovered until Q1 of the following year
- Forensic investigation takes months
- Regulatory investigations (HIPAA, GDPR, state AGs) take 12-36 months
- Class-action litigation can take 3-7 years to resolve

At any snapshot in time, your paid claims dramatically understate your true liability. The gap between what you have paid and what you will ultimately pay is called **IBNR: Incurred But Not Reported** (or more precisely, incurred but not yet fully developed).

**Your job: estimate ultimate losses from a triangle of partial data using the chain-ladder method.**

## The math

### The loss development triangle

Claims are organized into an **accident year × development age** triangle. Each cell $(y, d)$ shows the cumulative paid losses for accident year $y$ as of age $d$ months.

The diagonal of the triangle is where you stand today: accident year 1 is fully mature (you can see it at all development ages), but accident year 4 is only 12 months old — most of its losses are still IBNR.

```
         Dev age →  12mo   24mo   36mo   48mo
Accident year 1       X      X      X      X   ← fully developed
Accident year 2       X      X      X      ·   ← 3 data points
Accident year 3       X      X      ·      ·   ← 2 data points
Accident year 4       X      ·      ·      ·   ← 1 data point (today)
```

### Chain-ladder method

For each development age transition, compute the **age-to-age development factor** (also called a link ratio or LDF):

$$f_{d \to d+1} = \frac{\sum_y C_{y,d+1}}{\sum_y C_{y,d}}$$

where the sum is over all accident years where both columns are observed.

Apply the factors forward from each accident year's latest diagonal value to estimate the **ultimate loss** $C_{y,\text{ult}}$:

$$C_{y,\text{ult}} = C_{y,d^*} \times f_{d^* \to d^*+1} \times f_{d^*+1 \to d^*+2} \times \cdots \times f_{\text{tail}\to\infty}$$

The **IBNR reserve** for accident year $y$ is $C_{y,\text{ult}} - C_{y,d^*}$: the difference between the ultimate estimate and what has been paid so far.

### The tail factor

Even after 48 months, cyber claims are not always fully settled. The **tail factor** (from 48 months to ultimate) captures the remaining development beyond your data. In cyber, this tail is longer than most lines — a single regulatory investigation can take 5+ years. Underestimating the tail factor is one of the most common reserving errors in young lines.

In [ ]:
# Concept — development triangle anatomy and loss development pattern
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as _mpatches

fig, (ax_tri, ax_dev) = plt.subplots(1, 2, figsize=(13, 4.8),
                                      gridspec_kw={'width_ratios': [1.1, 1]})
fig.suptitle('Reserving Concepts: Triangle Structure and Loss Development',
             fontsize=12, fontweight='bold')

# ── Left: triangle anatomy grid ──────────────────────────────────────
_N    = 4
_CW, _CH = 1.6, 0.9
_DEV  = ['12mo', '24mo', '36mo', '48mo']
_AY   = ['AY 1\n(4 yrs)', 'AY 2\n(3 yrs)', 'AY 3\n(2 yrs)', 'AY 4\n(current)']

for _i in range(_N):
    for _j in range(_N):
        _obs  = _j <= (_N - 1 - _i)
        _diag = _j == (_N - 1 - _i)
        _x, _y = _j * _CW, (_N - 1 - _i) * _CH
        _fc = ('#aed6f1' if _diag else '#d6eaf8') if _obs else '#fadbd8'
        _ec = '#c0392b' if _diag else ('#5dade2' if _obs else '#e74c3c')
        _lw = 2.8 if _diag else 0.8
        ax_tri.add_patch(plt.Rectangle((_x, _y), _CW, _CH,
                                        fc=_fc, ec=_ec, lw=_lw, zorder=2))
        if _diag:
            ax_tri.text(_x + _CW/2, _y + _CH/2, 'TODAY \u25cf',
                        ha='center', va='center', fontsize=7.5,
                        color='#1a5276', fontweight='bold')
        elif _obs:
            ax_tri.text(_x + _CW/2, _y + _CH/2, 'paid\nclaims',
                        ha='center', va='center', fontsize=7, color='#1a5276')
        else:
            ax_tri.text(_x + _CW/2, _y + _CH/2, 'IBNR\n?',
                        ha='center', va='center', fontsize=8,
                        color='#922b21', fontweight='bold')

# Column headers
for _j, _lbl in enumerate(_DEV):
    ax_tri.text(_j * _CW + _CW/2, _N * _CH + 0.12, _lbl,
                ha='center', va='bottom', fontsize=9, fontweight='bold')
ax_tri.text(_N * _CW / 2, _N * _CH + 0.38, '\u2190 development age \u2192',
            ha='center', va='bottom', fontsize=8.5, color='#555')

# Row headers
for _i, _lbl in enumerate(_AY):
    ax_tri.text(-0.15, (_N - 1 - _i) * _CH + _CH/2, _lbl,
                ha='right', va='center', fontsize=8)

# Chain-ladder projection arrows for AY3 and AY4
for _i in [2, 3]:
    _j0  = _N - 1 - _i    # last observed column
    _y_m = (_N - 1 - _i) * _CH + _CH / 2
    for _jt in range(_j0 + 1, _N):
        ax_tri.annotate('',
            xy=((_jt + 0.82) * _CW, _y_m),
            xytext=((_jt + 0.18) * _CW, _y_m),
            arrowprops=dict(arrowstyle='->', color='#922b21',
                            lw=1.4, alpha=0.7))

ax_tri.text(_N * _CW * 0.62, -0.28,
            '\u2192 chain-ladder projects across the unknown cells',
            fontsize=7.5, color='#922b21', ha='center')

_legend_patches = [
    _mpatches.Patch(fc='#d6eaf8', ec='#5dade2', label='Observed paid losses'),
    _mpatches.Patch(fc='#aed6f1', ec='#c0392b', label='Current diagonal (latest data)'),
    _mpatches.Patch(fc='#fadbd8', ec='#e74c3c', label='IBNR — must be estimated'),
]
ax_tri.legend(handles=_legend_patches, loc='lower left',
              fontsize=7.5, framealpha=0.95)
ax_tri.set_xlim(-1.5, _N * _CW + 0.2)
ax_tri.set_ylim(-0.5, _N * _CH + 0.6)
ax_tri.set_title('Loss Development Triangle Structure', fontsize=10, fontweight='bold')
ax_tri.axis('off')

# ── Right: development pattern (% of ultimate paid over time) ────────
_ages_obs = np.array([12, 24, 36, 48])
_ages_ult = np.array([48, 60])          # 60mo = ultimate proxy
_pct_obs  = np.array([0.38, 0.64, 0.82, 0.93])
_pct_ult  = np.array([0.93, 1.00])

ax_dev.plot(_ages_obs, _pct_obs, 'o-', color='steelblue', linewidth=2.2,
            markersize=7, label='Observed development', zorder=3)
ax_dev.plot(_ages_ult, _pct_ult, 's--', color='#922b21', linewidth=2.2,
            markersize=7, label='Tail (estimated)', zorder=3)
ax_dev.fill_between(_ages_obs, _pct_obs, alpha=0.18, color='steelblue')
ax_dev.fill_between(_ages_ult, _pct_ult, alpha=0.20, color='#922b21')
ax_dev.fill_between([48, 60], [0.93, 0.93], [0.93, 1.00],
                    alpha=0.30, color='#922b21')

ax_dev.axvline(48, color='#c0392b', linestyle=':', linewidth=1.8,
               label='Edge of data (today)')
ax_dev.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax_dev.text(61, 1.01, 'Ultimate\n(100%)', fontsize=8, color='gray', va='bottom')

ax_dev.annotate('Tail factor bridges\nthis gap — the key\njudgment call',
                xy=(54, 0.965), xytext=(35, 0.73),
                fontsize=8.5, color='#922b21', ha='center',
                arrowprops=dict(arrowstyle='->', color='#922b21', lw=1.3),
                bbox=dict(boxstyle='round,pad=0.35', fc='white',
                          ec='#922b21', alpha=0.93))

for _age, _pct, _lbl in zip(_ages_obs, _pct_obs,
                              ['38%', '64%', '82%', '93%']):
    ax_dev.text(_age, _pct + 0.02, _lbl, ha='center', fontsize=8,
                color='#1a5276', fontweight='bold')

ax_dev.set_xlabel('Development age (months)', fontsize=10)
ax_dev.set_ylabel('Cumulative % of ultimate paid', fontsize=10)
ax_dev.set_title('Cyber Loss Development Pattern\nLosses are slow to emerge',
                  fontsize=10, fontweight='bold')
ax_dev.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax_dev.set_ylim(0.15, 1.18)
ax_dev.set_xlim(8, 68)
ax_dev.legend(fontsize=8.5, loc='lower right', framealpha=0.9)
ax_dev.grid(True, alpha=0.3)
ax_dev.spines['top'].set_visible(False)
ax_dev.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# Development ages in the triangle (months)
DEV_AGES    = [12, 24, 36, 48]
ACC_YEARS   = [1, 2, 3, 4]   # accident years (relative to Arclight founding)
N_YEARS     = len(ACC_YEARS)
N_DEV       = len(DEV_AGES)

# Earned premium by accident year (grows as book grows)
EARNED_PREMIUM = np.array([18_000_000, 32_000_000, 51_000_000, 66_000_000])

## Arclight's loss development triangle

The claims team has compiled four years of cumulative paid losses at each development age. Cells marked `—` are in the future — Arclight does not exist there yet.

In [ ]:
# Cumulative paid losses — the lower-right triangle is unobserved (NaN)
# True underlying ultimate losses (hidden from the player) are generated below
# fmt: (acc_year, [12mo, 24mo, 36mo, 48mo]) — NaN = not yet observable

# True development pattern: cyber losses develop slowly
# Cumulative % paid by dev age: 38%, 64%, 82%, 93%  (tail to ult: 1.075)
TRUE_DEV_PATTERN = np.array([0.38, 0.64, 0.82, 0.93])  # % of ultimate paid by each age
TRUE_TAIL_FACTOR = 1.075   # 48mo -> ultimate

# True ultimate loss ratios by accident year (mild adverse development in years 3-4)
TRUE_ULT_LR = np.array([0.595, 0.615, 0.650, 0.670])
TRUE_ULT    = EARNED_PREMIUM * TRUE_ULT_LR

# Build the full triangle from true values + development pattern
rng         = np.random.default_rng(42)
full_triangle = np.outer(TRUE_ULT, TRUE_DEV_PATTERN)
# Add modest process noise
noise = rng.normal(0, 0.015, full_triangle.shape)
full_triangle = full_triangle * (1 + noise)
full_triangle = np.maximum(full_triangle, 0)

# Observed triangle: upper-left only (diagonal = current state)
observed = full_triangle.copy()
for i in range(N_YEARS):
    for j in range(N_DEV):
        if j > (N_YEARS - 1 - i):  # future cells
            observed[i, j] = np.nan

# Display the triangle
col_labels = [f'{d}mo' for d in DEV_AGES]
row_labels = [f'AY {y}' for y in ACC_YEARS]

def fmt_cell(v):
    return f'${v/1e6:.2f}M' if not np.isnan(v) else '—'

disp_data = [[fmt_cell(observed[i, j]) for j in range(N_DEV)] for i in range(N_YEARS)]
triangle_disp = pd.DataFrame(disp_data, index=row_labels, columns=col_labels)

print('Cumulative paid losses (upper-left triangle = observed data)')
print()
display(triangle_disp)
print()
print('Earned premium by accident year:')
for i, (yr, ep) in enumerate(zip(ACC_YEARS, EARNED_PREMIUM)):
    print(f'  AY {yr}: ${ep/1e6:.1f}M  '
          f'(reported loss ratio at latest diagonal: '
          f'{observed[i, N_YEARS - 1 - i] / ep:.1%})')

## Step 1: Calculate the age-to-age factors

For each column-to-column transition, compute the weighted-average development factor using the accident years where both columns are observable.

In [ ]:
def compute_ldfs(triangle):
    """Compute volume-weighted age-to-age LDFs from the observed triangle."""
    n_rows, n_cols = triangle.shape
    ldfs = []
    for j in range(n_cols - 1):
        # Use rows where both col j and col j+1 are observed
        mask      = ~np.isnan(triangle[:, j]) & ~np.isnan(triangle[:, j+1])
        numerator = triangle[mask, j + 1].sum()
        denom     = triangle[mask, j].sum()
        ldfs.append(numerator / denom if denom > 0 else np.nan)
    return np.array(ldfs)


ldfs = compute_ldfs(observed)

print('Age-to-age development factors (LDFs):')
for i, (from_age, to_age, ldf) in enumerate(
        zip(DEV_AGES[:-1], DEV_AGES[1:], ldfs)):
    print(f'  {from_age}mo -> {to_age}mo:  {ldf:.4f}')

# Cumulative development factors (CDF to ultimate, excluding tail)
cdfs_no_tail = np.ones(len(ldfs) + 1)
for i in range(len(ldfs) - 1, -1, -1):
    cdfs_no_tail[i] = cdfs_no_tail[i + 1] * ldfs[i]

print()
print('Cumulative development factors (CDF to 48mo ultimate — before tail factor):')
for age, cdf in zip(DEV_AGES, cdfs_no_tail):
    pct_paid = 1 / cdf
    print(f'  {age:2d}mo: CDF = {cdf:.4f}  ({pct_paid:.1%} of ultimate paid)')

## Step 2: Set the tail factor

The data only goes to 48 months. But cyber claims — especially breach notification, regulatory fines, and class-action suits — can run 5-7 years. The **tail factor** multiplies the 48-month CDF to extend the development to ultimate.

A tail factor of 1.0 assumes all losses are fully paid by 48 months. A tail factor of 1.10 means 10% more development is expected beyond the data.

Adjust the tail factor and observe how sensitive the IBNR reserve is to this assumption — this is the single largest source of reserving uncertainty in cyber.

In [ ]:
tail_slider = widgets.FloatSlider(
    value=1.00, min=1.00, max=1.30, step=0.005,
    description='Tail factor:',
    readout_format='.3f',
    layout=widgets.Layout(width='500px')
)
reserve_output = widgets.Output()


def update_reserves(change):
    tail = tail_slider.value

    # Full CDFs including tail
    cdfs = cdfs_no_tail * tail

    # Project ultimates from each accident year's latest diagonal
    ultimates  = np.zeros(N_YEARS)
    ibne       = np.zeros(N_YEARS)   # incurred but not enough (total reserve)
    reported   = np.zeros(N_YEARS)
    for i in range(N_YEARS):
        latest_age_idx = N_YEARS - 1 - i   # index of latest observed dev age
        latest_val     = observed[i, latest_age_idx]
        ultimates[i]   = latest_val * cdfs[latest_age_idx]
        reported[i]    = latest_val
        ibne[i]        = ultimates[i] - latest_val

    total_reported  = reported.sum()
    total_ultimate  = ultimates.sum()
    total_ibnr      = ibne.sum()
    true_ultimate   = TRUE_ULT.sum()
    reserve_error   = total_ultimate - true_ultimate  # + = over-reserved

    with reserve_output:
        reserve_output.clear_output(wait=True)

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        # Stacked bar: paid vs IBNR by accident year
        x = np.arange(N_YEARS)
        axes[0].bar(x, reported / 1e6, color='steelblue', label='Paid to date')
        axes[0].bar(x, ibne / 1e6, bottom=reported / 1e6,
                    color='lightsalmon', edgecolor='crimson', linewidth=0.8, label='IBNR reserve')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels([f'AY {y}' for y in ACC_YEARS])
        axes[0].set_ylabel('$M')
        axes[0].set_title('Paid vs. IBNR reserve by accident year')
        axes[0].legend(fontsize=8)

        # Development projection: AY1 and AY4
        ages_full = np.array(DEV_AGES + [60])   # 60mo = ultimate proxy
        for i, color, label in [(0, 'steelblue', 'AY 1 (mature)'),
                                 (3, 'crimson',   'AY 4 (immature)')]:
            obs_vals = [observed[i, j] for j in range(N_DEV) if not np.isnan(observed[i, j])]
            obs_ages = [DEV_AGES[j] for j in range(N_DEV) if not np.isnan(observed[i, j])]
            axes[1].plot(obs_ages, [v / 1e6 for v in obs_vals],
                         'o-', color=color, linewidth=2, label=label)
            # Project forward
            latest_idx = N_YEARS - 1 - i
            proj_ages  = DEV_AGES[latest_idx:] + [60]
            proj_cdf   = cdfs[latest_idx:]
            proj_vals  = [observed[i, latest_idx] * cdfs[latest_idx] / c
                          for c in np.append(proj_cdf, 1.0)]  # walk forward
            # Simpler: walk using LDFs
            proj_pts = [observed[i, latest_idx]]
            for j in range(latest_idx, N_DEV - 1):
                proj_pts.append(proj_pts[-1] * ldfs[j])
            proj_pts.append(proj_pts[-1] * tail)  # tail
            axes[1].plot(proj_ages, [v / 1e6 for v in proj_pts],
                         's--', color=color, alpha=0.6, linewidth=1.5)
        axes[1].set_xlabel('Development age (months)')
        axes[1].set_ylabel('Cumulative paid losses ($M)')
        axes[1].set_title('Loss development projection')
        axes[1].legend(fontsize=8)
        axes[1].grid(True, alpha=0.3)

        # Tail factor sensitivity: IBNR vs tail factor
        tail_range  = np.linspace(1.0, 1.30, 100)
        ibnr_range  = []
        for tf in tail_range:
            cdfs_t = cdfs_no_tail * tf
            total  = sum(
                observed[i, N_YEARS - 1 - i] * cdfs_t[N_YEARS - 1 - i] - observed[i, N_YEARS - 1 - i]
                for i in range(N_YEARS)
            )
            ibnr_range.append(total / 1e6)
        axes[2].plot(tail_range, ibnr_range, 'b-', linewidth=2)
        axes[2].axvline(tail, color='crimson', linestyle='--', linewidth=1.5,
                        label=f'Your tail: {tail:.3f}')
        axes[2].axvline(TRUE_TAIL_FACTOR, color='seagreen', linestyle=':', linewidth=1.5,
                        label=f'True tail: {TRUE_TAIL_FACTOR:.3f}')
        axes[2].set_xlabel('Tail factor')
        axes[2].set_ylabel('Total IBNR reserve ($M)')
        axes[2].set_title('Reserve sensitivity to tail factor')
        axes[2].legend(fontsize=8)
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Summary table
        summary_rows = []
        for i in range(N_YEARS):
            lr_reported = reported[i] / EARNED_PREMIUM[i]
            lr_ultimate = ultimates[i] / EARNED_PREMIUM[i]
            summary_rows.append({
                'Accident Year': f'AY {ACC_YEARS[i]}',
                'Earned Premium': f'${EARNED_PREMIUM[i]/1e6:.1f}M',
                'Paid to Date':   f'${reported[i]/1e6:.2f}M',
                'Reported LR':    f'{lr_reported:.1%}',
                'IBNR Reserve':   f'${ibne[i]/1e6:.2f}M',
                'Ultimate Est.':  f'${ultimates[i]/1e6:.2f}M',
                'Ultimate LR':    f'{lr_ultimate:.1%}',
            })
        summary_df = pd.DataFrame(summary_rows)
        display(summary_df.to_html(index=False))

        lines = []
        lines.append(f'<p><b>Tail factor: {tail:.3f} | '
                     f'Total IBNR: ${total_ibnr/1e6:.2f}M | '
                     f'Total ultimate: ${total_ultimate/1e6:.2f}M</b></p>')
        lines.append(f'<p>Overall ultimate loss ratio: '
                     f'<b>{total_ultimate / EARNED_PREMIUM.sum():.1%}</b></p>')

        if abs(reserve_error / true_ultimate) < 0.03:
            lines.append(f'<p style="color:seagreen"><b>Accurate reserve.</b> '
                         f'Your ultimate estimate (${total_ultimate/1e6:.2f}M) is within 3% '
                         f'of the true ultimate (${true_ultimate/1e6:.2f}M). '
                         f'Well-chosen tail factor.</p>')
        elif reserve_error < 0:
            shortfall = -reserve_error / 1e6
            lines.append(f'<p style="color:crimson"><b>Under-reserved by ${shortfall:.2f}M.</b> '
                         f'Your IBNR is too low. The balance sheet overstates profitability. '
                         f'Try increasing the tail factor — cyber litigation runs long.</p>')
        else:
            excess = reserve_error / 1e6
            lines.append(f'<p style="color:darkorange"><b>Over-reserved by ${excess:.2f}M.</b> '
                         f'Your IBNR is conservative. Capital is tied up unnecessarily. '
                         f'Try reducing the tail factor.</p>')

        lines.append(f'<p>The <b>true tail factor is revealed</b> in the sensitivity chart '
                     f'(green dotted line). Notice how a change of even 0.05 in the tail '
                     f'factor moves the total IBNR reserve by '
                     f'${abs((ibnr_range[-1] - ibnr_range[0]) * 0.05 / 0.30):.1f}M.</p>')

        lines.append('<hr><p><b>Key takeaway.</b> '
                     'Reserves are estimates, not facts. The chain-ladder method is mechanical '
                     'and transparent, but it depends critically on the tail factor — a judgment '
                     'call that can move total IBNR by millions. '
                     'In a young line like cyber, there is limited historical data to anchor '
                     'the tail, so actuaries lean on industry benchmarks, coverage analysis, '
                     'and case-by-case large-loss reviews. '
                     'Under-reserving is the silent killer: it makes every year look profitable '
                     'until the bills come due.</p>')
        display(HTML(''.join(lines).replace('$', '&#36;')))


tail_slider.observe(update_reserves, names='value')
display(widgets.VBox([tail_slider, reserve_output]))
update_reserves(None)

## Step 3: Diagnostic — are the LDFs stable?

A key assumption of the chain-ladder method is that the age-to-age development pattern is stable across accident years. If the most recent years show faster or slower development than older years, it may signal a trend — for example, tightening litigation environments or improving claims handling speed.

The **diagnostic triangle** shows the individual year-to-year ratios behind the weighted averages.

In [ ]:
print('Individual age-to-age ratios by accident year (weighted average in parentheses):')
print()

col_headers = [f'{DEV_AGES[j]}mo-{DEV_AGES[j+1]}mo' for j in range(N_DEV - 1)]
diag_rows   = []
for i in range(N_YEARS):
    row = {'Accident Year': f'AY {ACC_YEARS[i]}'}
    for j in range(N_DEV - 1):
        if (not np.isnan(observed[i, j])) and (not np.isnan(observed[i, j + 1])):
            ratio = observed[i, j + 1] / observed[i, j]
            row[col_headers[j]] = f'{ratio:.4f}'
        else:
            row[col_headers[j]] = '—'
    diag_rows.append(row)

# Add weighted average row
avg_row = {'Accident Year': 'Weighted avg'}
for j, hdr in enumerate(col_headers):
    avg_row[hdr] = f'{ldfs[j]:.4f}'
diag_rows.append(avg_row)

diag_df = pd.DataFrame(diag_rows)
display(diag_df.to_html(index=False))

print()
print('If individual ratios vary widely within a column, the weighted average may be unstable.')
print('Cyber is a rapidly evolving line — a 2-year-old development pattern may not reflect')
print('current litigation trends or coverage interpretations.')

## Hints

<details><summary>Hint 1 — how to read the triangle</summary>

Each row is one accident year. Moving right shows how cumulative paid losses grow as the claims age. AY 1 at 12 months shows what had been paid in the first year; AY 1 at 48 months shows what had been paid over four years for the same group of claims. The ratio between adjacent columns is the LDF.

</details>

<details><summary>Hint 2 — finding the true tail</summary>

The sensitivity chart shows a green dotted line at the true tail factor. Drag the slider until your reserve matches the true ultimate within a few percent. Notice how small the tail factor change is that separates a well-reserved from a dangerously under-reserved position.

</details>

<details><summary>Hint 3 — why AY 4 is the most uncertain</summary>

AY 4 has only 12 months of data. At 12 months, only 38% of ultimate losses are paid — the chain-ladder must project the remaining 62% using factors derived from older years. A small error in the 12mo-to-24mo LDF compounds through all subsequent factors.

</details>

<details><summary>Hint 4 — cyber-specific tail considerations</summary>

Regulatory proceedings (HIPAA OCR, state AGs, GDPR DPAs) have their own timelines independent of claims payment. A breach from year 1 may still have an open regulatory matter in year 5. Class actions can take 7+ years. Cyber insurers with significant healthcare and financial services books often use tail factors of 1.10–1.25, well above what the data triangle alone would imply.

</details>

## What's next

**Chapter 6 — Sharing the Risk (Cyber Reinsurance).** Arclight has reserves, a rate manual, and a growing book. But Chapter 2 showed that a systemic event can blow through an entire year of premiums in one go. The solution is reinsurance: transfer part of the tail risk to a reinsurer in exchange for a portion of the premium.

Chapter 6 introduces the two main reinsurance structures — **quota share** (cede a fixed percentage of every risk) and **excess of loss** (the reinsurer pays only above a retention threshold). You will price both structures using the lognormal model from Chapter 3 and Limited Expected Value, then decide which structure best protects Arclight against a systemic attack while leaving enough premium on the table to be profitable.